In [19]:
import os
import numpy as np
import pandas as pd

cycle = 2016

file_path = './2012-2022/raw/house_2016.csv'

df = pd.read_csv(file_path, dtype=str, encoding="latin-1")
df.columns = df.columns.str.strip()

print('shape initial: ', np.shape(df))

inclusion = ['FEC ID#', 'GENERAL VOTES']

df.dropna(subset=inclusion, inplace=True)
df = df[~df['D'].str.contains('UNEXPIRED')]

keep = ['STATE ABBREVIATION', 'D', 'FEC ID#', 'CANDIDATE NAME (First)', 'CANDIDATE NAME (Last)', 'CANDIDATE NAME', 'PARTY', 'GENERAL VOTES', 'GENERAL %', 'GE WINNER INDICATOR']

df = df[keep]

print('shape after filtering on general elections and dropna on essential cols:' , np.shape(df))

df.columns = df.columns.str.strip()

df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)

df = df.rename(columns={
    "FEC ID#": "cand_id",
    "STATE ABBREVIATION": "state",
    'CANDIDATE NAME (First)': 'cand_name_first',
    'CANDIDATE NAME (Last)': 'cand_name_last',
    'PARTY':'party',
    "D":"district",
    "GENERAL VOTES":"votes",
    "GENERAL %": "vote_share",
    'GE WINNER INDICATOR': 'outcome'
})

df['cycle'] = cycle
df["state"] = df["state"].str.upper().str.strip()
df["district"] = df["district"].str.extract(r"(\d+)", expand=False).fillna("0").astype(int)
df['party'] = df['party'].replace({'R':'REP', 'D':'DEM'})
df["votes_raw"] = df["votes"].str.strip()
df["unopposed"] = df["votes_raw"].eq("Unopposed")
df["votes"] = pd.to_numeric(df["votes_raw"].str.replace(",", "", regex=False), errors="coerce").astype("Int64")

df["vote_share"] = df["vote_share"].str.replace("%", "", regex=False).astype(float)

if df["vote_share"].max() > 1.5:
    df["vote_share"] /= 100

df.loc[df["unopposed"], "vote_share"] = 1.0
df.drop(columns=['votes_raw'], inplace=True)

df["outcome"] = (df["outcome"] == "W").astype(int)

ordered_cols = ['cand_id', 'cycle', 'state', 'district', 'cand_name_first', 'cand_name_last', 'party', 'votes', 'vote_share', 'outcome', 'unopposed']

df = df[ordered_cols].reset_index(drop=True)

print('shape after dropping write-ins, misc, and ultra low vote-share cands: ', np.shape(df))

df.head(30)


shape initial:  (4131, 23)
shape after filtering on general elections and dropna on essential cols: (1279, 10)
shape after dropping write-ins, misc, and ultra low vote-share cands:  (1279, 11)


,cand_id,cycle,state,district,cand_name_first,cand_name_last,party,votes,vote_share,outcome,unopposed
0,H4AL01123,2016,AL,1,Bradley,Byrne,REP,208083,0.963825,1,False
1,H0AL02087,2016,AL,2,Martha,Roby,REP,134886,0.487685,1,False
2,H6AL02167,2016,AL,2,Nathan,Mathis,DEM,112089,0.405262,0,False
3,H2AL03032,2016,AL,3,Mike,Rogers,REP,192164,0.669318,1,False
4,H4AL03061,2016,AL,3,Jesse,Smith,DEM,94549,0.329320,0,False
5,H6AL04098,2016,AL,4,Robert,Aderholt,REP,235925,0.985303,1,False
6,H0AL05163,2016,AL,5,Mo,Brooks,REP,205647,0.666979,1,False
7,H6AL05202,2016,AL,5,"Will, Jr.",Boyd,DEM,102234,0.331578,0,False
8,H4AL06098,2016,AL,6,Gary,Palmer,REP,245313,0.744939,1,False
9,H6AL06127,2016,AL,6,David J.,Putman,DEM,83709,0.254198,0,False


In [20]:
voteshare = df.groupby(["state", "district"])["vote_share"].sum()

bad_groups = voteshare[(voteshare < 0.95) | (voteshare > 1.05)]

display(bad_groups)

state  district
AL     2           0.892948
PA     7           0.405174
VA     11          0.877785
Name: vote_share, dtype: float64

In [21]:
df[df['state']=='PA'][df['district']==7]

C:\Users\Paul\AppData\Local\Temp\ipykernel_21740\1253566321.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  df[df['state']=='PA'][df['district']==7]


,cand_id,cycle,state,district,cand_name_first,cand_name_last,party,votes,vote_share,outcome,unopposed
990,H0PA07082,2016,PA,7,Patrick L.,Meehan,REP,225678,NaN,1,False
991,H4PA07092,2016,PA,7,Mary Ellen,Balchunis,DEM,153824,0.405174,0,False


In [22]:
def infer_vote_share(df: pd.DataFrame) -> pd.DataFrame:
    contest_keys = ["state", "district"]
    contest_totals = df.groupby(contest_keys)["votes"].transform("sum")

    missing = df["vote_share"].isna() & df["votes"].notna() & contest_totals.gt(0)

    df.loc[missing, "vote_share"] = df.loc[missing, "votes"] / contest_totals[missing]

    return df

new_df = infer_vote_share(df)

new_df[df['state']=='PA'][df['district']==7]

C:\Users\Paul\AppData\Local\Temp\ipykernel_21740\2840458136.py:13: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  new_df[df['state']=='PA'][df['district']==7]


,cand_id,cycle,state,district,cand_name_first,cand_name_last,party,votes,vote_share,outcome,unopposed
990,H0PA07082,2016,PA,7,Patrick L.,Meehan,REP,225678,0.594669,1,False
991,H4PA07092,2016,PA,7,Mary Ellen,Balchunis,DEM,153824,0.405174,0,False


In [23]:
voteshare = new_df.groupby(["state", "district"])["vote_share"].sum()

bad_groups = voteshare[(voteshare < 0.95) | (voteshare > 1.05)]

display(bad_groups)

state  district
AL     2           0.892948
VA     11          0.877785
Name: vote_share, dtype: float64

In [25]:
new_df.to_csv('2012-2022/clean/house_2016.csv', index=False)